### $$\textbf{Ramdeobaba University}$$

### $$\textbf{Department of Computer Science and Engineering}$$

### $$\textbf{Software Laboratory – I (24CS01PR0303)}$$

### $$\textbf{Group Project}$$

## 👥 Team Members

| # | Name | Roll Number | Batch |
|---|------|-------------|---------------------|
| 1 | Umair Khan | A12-B3-50 | B-3 |
| 2 | Utkarsh Vijay Kumbhare | A12-B3-37 | B-3 |
| 3 | Mohammad Afham | A12-B4-60 | B-4 |
  
**Submission Date: 15th October**

**Project Topic: Student Result Analyzer**

---

## 📖 Introduction

The **Student Result Analyzer** is a Python-based application that reads student
marks from a text file, performs statistical analysis using **NumPy**, assigns
grades, and generates a formatted report — all automatically.

Manual result processing is slow, error-prone, and doesn't scale. This project
demonstrates how core Python data structures and NumPy can be combined to build
a clean, modular analysis pipeline.

---

## 🎯 Objectives

- Read student records from an external `.txt` file
- Store data using **nested dictionaries** and **tuples**
- Compute total marks, averages, and identify the class topper
- Perform **subject-wise statistics** (mean, max, min, 75th percentile)
- Assign grades based on average marks
- Identify **unique grades** across the class (using sets)
- Generate and save a formatted report to a file

---

## 🧠 Concepts Covered (Mapped to Labs 1–5)

| Lab | Concept | Usage in Project |
|-----|---------|------------------|
| Lab 1 | Input/Output, `if-elif-else`, loops | Grade logic, iterating records |
| Lab 2 | Strings, Lists, Tuples, slicing | `split()`, `strip()`, marks tuple |
| Lab 3 | Sets, Nested Dictionaries | `unique_grades()`, `students{}` |
| Lab 4 | Functions, File Handling, `os` | Modular functions, `open()`, `os.path` |
| Lab 5 | NumPy arrays, statistics, boolean masks | `np.mean`, `np.max`, `np.percentile`, `col[col > 90]` |

---

## 📂 Input / Output

**Input** — `student_data.txt` (20 records):

---

## Environment Setup


In [2]:
import os
import sys
import numpy as np

INPUT_FILE  = "student_data.txt"
OUTPUT_FILE = "result_report.txt"

print("Python :", sys.version.split()[0])
print("NumPy  :", np.__version__)
print("Folder :", os.getcwd())
print("Files  :", [f for f in os.listdir() if not f.startswith('.')])

Python : 3.13.9
NumPy  : 2.3.5
Folder : C:\Users\Umair khan\Desktop\sl-lab project\StudentResultAnalyzer
Files  : ['main.ipynb', 'README.md', 'student_data.txt', 'venv']


## Reading Data

In [3]:
def read_student_data(filename):
    """Read the marks file and return {roll: {'name': ..., 'marks': (...)}}

    Concepts:
      - File Handling  : open(), with, read line-by-line
      - Strings        : strip(), split()
      - Lists & Tuples : parts list, marks tuple
      - Nested Dict    : {roll: {name, marks}}
    """
    students = {}

    if not os.path.exists(filename):
        print(f"[ERROR] File '{filename}' not found.")
        return students

    with open(filename, "r") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue

            parts = [p.strip() for p in line.split(",")]

            if len(parts) != 5:
                print(f"[WARN] Skipping malformed line: {line}")
                continue

            roll  = int(parts[0])
            name  = parts[1]
            marks = tuple(int(m) for m in parts[2:])   # tuple

            students[roll] = {"name": name, "marks": marks}

    return students

In [5]:
# --- Test Case ---
test = read_student_data(INPUT_FILE)
print("Records loaded:", len(test))
print("Sample record :", test[101])
print("Last record   :", test[120])

Records loaded: 20
Sample record : {'name': 'Amit', 'marks': (85, 90, 88)}
Last record   : {'name': 'Ishita', 'marks': (93, 91, 95)}


## Find Totals , Topper

In [6]:
def calculate_total_and_average(marks_tuple):
    """Return (total, average) using NumPy."""
    arr = np.array(marks_tuple)
    return int(np.sum(arr)), round(float(np.mean(arr)), 2)


def find_topper(students):
    """Return (roll, total) of the highest-scoring student."""
    topper_roll, topper_total = None, -1
    for roll, info in students.items():
        total, _ = calculate_total_and_average(info["marks"])
        if total > topper_total:
            topper_total, topper_roll = total, roll
    return topper_roll, topper_total

In [7]:
# --- Test Case ---
print("Roll 114 (96,94,98):", calculate_total_and_average((96, 94, 98)))
print("Roll 101 (85,90,88):", calculate_total_and_average((85, 90, 88)))

students = read_student_data(INPUT_FILE)
print("Topper :", find_topper(students))

Roll 114 (96,94,98): (288, 96.0)
Roll 101 (85,90,88): (263, 87.67)
Topper : (114, 288)


## Subject Statistics + Grade Logic

In [8]:
def subject_wise_statistics(students):
    """Return {subject: {mean, max, min, percentile, above_90}}"""
    marks_matrix = np.array([info["marks"] for info in students.values()])
    subjects = ["Physics", "Chemistry", "Maths"]
    stats = {}

    for i, sub in enumerate(subjects):
        col = marks_matrix[:, i]
        stats[sub] = {
            "mean":       round(float(np.mean(col)), 2),
            "max":        int(np.max(col)),
            "min":        int(np.min(col)),
            "percentile": round(float(np.percentile(col, 75)), 2),
            "above_90":   int(col[col > 90].size),   # boolean mask
        }
    return stats


def assign_grade(average):
    """Return grade letter for a given average (if/elif/else)."""
    if average >= 90:   return "A+"
    elif average >= 80: return "A"
    elif average >= 70: return "B"
    elif average >= 60: return "C"
    elif average >= 50: return "D"
    else:               return "F"

In [9]:
# --- Test Case ---
students = read_student_data(INPUT_FILE)
stats = subject_wise_statistics(students)

print("Subject-wise statistics:")
for sub, s in stats.items():
    print(f"  {sub:<10} → {s}")

print("\nGrade checks:")
print("  avg 96 →", assign_grade(96))
print("  avg 85 →", assign_grade(85))
print("  avg 75 →", assign_grade(75))
print("  avg 65 →", assign_grade(65))
print("  avg 55 →", assign_grade(55))
print("  avg 40 →", assign_grade(40))

Subject-wise statistics:
  Physics    → {'mean': 80.3, 'max': 96, 'min': 55, 'percentile': 90.25, 'above_90': 5}
  Chemistry  → {'mean': 82.75, 'max': 95, 'min': 60, 'percentile': 90.25, 'above_90': 5}
  Maths      → {'mean': 82.1, 'max': 98, 'min': 58, 'percentile': 91.25, 'above_90': 6}

Grade checks:
  avg 96 → A+
  avg 85 → A
  avg 75 → B
  avg 65 → C
  avg 55 → D
  avg 40 → F


## Grade Distribution + Unique Grades (Sets)

In [10]:
def grade_distribution(students):
    """Return {grade: count} across the whole class."""
    dist = {}
    for info in students.values():
        _, avg = calculate_total_and_average(info["marks"])
        g = assign_grade(avg)
        dist[g] = dist.get(g, 0) + 1
    return dist


def unique_grades(students):
    """Return the SET of distinct grades present in the class."""
    return {assign_grade(calculate_total_and_average(i["marks"])[1])
            for i in students.values()}

In [11]:
# --- Test Case ---
students = read_student_data(INPUT_FILE)
dist = grade_distribution(students)
grades = unique_grades(students)

print("Grade distribution :", dist)
print("Unique grades (set):", grades)
print("Sorted unique      :", sorted(grades))
print("Total students     :", sum(dist.values()))

Grade distribution : {'A': 6, 'A+': 6, 'B': 5, 'C': 2, 'D': 1}
Unique grades (set): {'A+', 'A', 'C', 'B', 'D'}
Sorted unique      : ['A', 'A+', 'B', 'C', 'D']
Total students     : 20


## Writing Report

In [12]:
def write_report(students, topper, stats, dist, grades):
    """Write formatted analysis to OUTPUT_FILE."""
    with open(OUTPUT_FILE, "w") as f:
        line = "=" * 62
        f.write(line + "\n")
        f.write("           STUDENT RESULT ANALYSIS REPORT\n")
        f.write(line + "\n\n")

        # Student-wise results
        f.write(f"{'Roll':<6}{'Name':<12}{'Phy':<6}{'Chem':<6}"
                f"{'Math':<6}{'Total':<8}{'Avg':<8}{'Grade'}\n")
        f.write("-" * 62 + "\n")

        for roll, info in sorted(students.items()):
            total, avg = calculate_total_and_average(info["marks"])
            grade = assign_grade(avg)
            phy, chem, math = info["marks"]
            f.write(f"{roll:<6}{info['name']:<12}{phy:<6}{chem:<6}"
                    f"{math:<6}{total:<8}{avg:<8}{grade}\n")

        # Class topper
        f.write("\n" + line + "\n")
        f.write(f"CLASS TOPPER : Roll {topper[0]} "
                f"({students[topper[0]]['name']}) — Total {topper[1]}\n")
        f.write(line + "\n\n")

        # Subject-wise statistics
        f.write("SUBJECT-WISE STATISTICS\n")
        f.write("-" * 62 + "\n")
        f.write(f"{'Subject':<12}{'Mean':<10}{'Max':<8}{'Min':<8}"
                f"{'75th %ile':<12}{'Above 90'}\n")
        for sub, s in stats.items():
            f.write(f"{sub:<12}{s['mean']:<10}{s['max']:<8}{s['min']:<8}"
                    f"{s['percentile']:<12}{s['above_90']}\n")

        # Grade distribution
        f.write("\nGRADE DISTRIBUTION\n")
        f.write("-" * 62 + "\n")
        for g in sorted(dist):
            bar = "#" * dist[g]
            f.write(f"  {g:<4}: {dist[g]:>2} student(s)  {bar}\n")

        # Unique grades (set)
        f.write("\nUNIQUE GRADES PRESENT : ")
        f.write(", ".join(sorted(grades)) + "\n")
        f.write("\n[Report generated successfully]\n")

    print(f"[OK] Report saved to: {OUTPUT_FILE}")

In [13]:
# --- Test Case ---
students = read_student_data(INPUT_FILE)
topper   = find_topper(students)
stats    = subject_wise_statistics(students)
dist     = grade_distribution(students)
grades   = unique_grades(students)

write_report(students, topper, stats, dist, grades)

[OK] Report saved to: result_report.txt


## File Status

In [14]:
print(open("result_report.txt").read())

           STUDENT RESULT ANALYSIS REPORT

Roll  Name        Phy   Chem  Math  Total   Avg     Grade
--------------------------------------------------------------
101   Amit        85    90    88    263     87.67   A
102   Riya        92    95    91    278     92.67   A+
103   Rahul       75    80    78    233     77.67   B
104   Sneha       88    84    90    262     87.33   A
105   Karan       65    70    72    207     69.0    C
106   Priya       95    93    97    285     95.0    A+
107   Arjun       78    82    80    240     80.0    A
108   Neha        82    88    85    255     85.0    A
109   Vikram      70    75    68    213     71.0    B
110   Ananya      91    89    94    274     91.33   A+
111   Rohit       60    65    58    183     61.0    C
112   Divya       87    90    92    269     89.67   A
113   Siddharth   73    78    75    226     75.33   B
114   Meera       96    94    98    288     96.0    A+
115   Aditya      68    72    70    210     70.0    B
116   Kavya       84  

## Main Driver

In [15]:
def main():
    """Main driver — reads data, computes everything, prints summary, writes file."""
    students = read_student_data(INPUT_FILE)
    if not students:
        print("No data to analyze. Exiting.")
        return

    print(f"Loaded {len(students)} student records.\n")

    topper = find_topper(students)
    stats  = subject_wise_statistics(students)
    dist   = grade_distribution(students)
    grades = unique_grades(students)

    print("=" * 55)
    print("            RESULT SUMMARY")
    print("=" * 55)
    print(f"Class Topper : {students[topper[0]]['name']} "
          f"(Roll {topper[0]}, Total {topper[1]})")

    print("\nSubject-wise Mean:")
    for sub, s in stats.items():
        print(f"  {sub:<10}: {s['mean']}")

    print("\nGrade Distribution:")
    for g in sorted(dist):
        print(f"  {g:<4}: {dist[g]} student(s)")

    print(f"\nUnique Grades : {sorted(grades)}")
    print("=" * 55)

    write_report(students, topper, stats, dist, grades)

In [16]:
main()

Loaded 20 student records.

            RESULT SUMMARY
Class Topper : Meera (Roll 114, Total 288)

Subject-wise Mean:
  Physics   : 80.3
  Chemistry : 82.75
  Maths     : 82.1

Grade Distribution:
  A   : 6 student(s)
  A+  : 6 student(s)
  B   : 5 student(s)
  C   : 2 student(s)
  D   : 1 student(s)

Unique Grades : ['A', 'A+', 'B', 'C', 'D']
[OK] Report saved to: result_report.txt
